# IMDB sentiment data tutorial

How **conditional** IMDB reviews are represented for next-token training in this repo.

Format (tags style, default):

```text
<bos>[SENTIMENT] positive [/SENTIMENT] [REVIEW] ...review text... [/REVIEW]<eos>
```

The training loop (`nano_llm.train`) calls **`load_imdb_sentiment`** and **`build_tokenizer_from_text`** for you. This notebook shows the pieces explicitly.

- Load raw IMDB via Hugging Face `datasets`
- Format with **`format_imdb_example`** from `nano_llm.data` (tags or natural instructions)
- Fit **`HFByteBPETokenizer`** on training text
- See how `create_dataloaders` batches for the model

## 0) Setup

```bash
pip install datasets tokenizers
```

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent]
repo_root = next((p for p in candidates if (p / "src" / "nano_llm").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repo root containing src/nano_llm")
sys.path.insert(0, str(repo_root / "src"))

from datasets import load_dataset

from nano_llm.data import create_dataloaders, format_imdb_example, load_imdb_sentiment
from nano_llm.tokenizer import HFByteBPETokenizer

## 1) Load IMDB (or use the built-in loader)

**Production training** uses `load_imdb_sentiment(...)` (same splits and normalization as `train.py`). Below we also show raw `datasets` if you want to inspect the HF object.

In [ ]:
ds = load_dataset("imdb")
print(ds)
print("train:", len(ds["train"]), "test:", len(ds["test"]))

In [ ]:
# Same formatting + normalization as training (subset for speed)
train_samples, val_samples = load_imdb_sentiment(
    max_train_samples=800,
    max_val_samples=200,
    imdb_conditioning_style="tags",
)
print("num train samples:", len(train_samples))
print("num val samples  :", len(val_samples))
print(train_samples[0][:200], "...")

## 2) Formatting API (`format_imdb_example`)

Each HF row becomes one or more strings (chunks) if `max_review_chars` is set.

In [ ]:
row = ds["train"][0]
one = format_imdb_example(row["text"], row["label"], imdb_conditioning_style="tags")
print("chunks:", len(one))
print(one[0][:240], "...")

## 3) Train the byte-level BPE tokenizer on train+val text

Join with newlines so examples stay separated. Vocab sizes like `256` or `8000` match common `scripts/train.py` settings.

In [ ]:
corpus = "\n".join(train_samples + val_samples)
tok = HFByteBPETokenizer.from_text(corpus, vocab_size=512, word_boundary_aware=False)
print("type:", tok.to_state()["type"], "vocab_size:", tok.vocab_size)
sample = train_samples[0]
ids = tok.encode(sample)
print("token count:", len(ids))
# Byte BPE may normalize spaces; compare decoded text to original
dec = tok.decode(ids)
print("exact roundtrip:", dec == sample)

## 4) `create_dataloaders` (next-token batches)

This matches what `train.py` uses after the tokenizer exists: chunked sequences, padding with ignore index `-100` on targets.

In [ ]:
train_loader, val_loader = create_dataloaders(
    train_samples,
    val_samples,
    tok,
    seq_len=128,
    batch_size=4,
)
batch = next(iter(train_loader))
x, y = batch[0], batch[1]
print("x shape:", tuple(x.shape), "y shape:", tuple(y.shape))

## 5) Train for real

```bash
python scripts/train.py --tokenizer-type hf_bpe_byte --bpe-vocab-size 256 \
  --imdb-max-review-chars 500 --epochs 10 --checkpoint-dir checkpoints/imdb_demo
```

Docker: `make train-imdb` (see `README.md`).

**Natural instructions** instead of tags: `--imdb-conditioning-style natural` (must match at generation time; see `scripts/generate.py` / `scripts/chat.py`).

## 6) Generation prompt shape

At inference, start from the same prefix the model saw during training, e.g.:

```text
<bos>[SENTIMENT] positive [/SENTIMENT] [REVIEW] 
```

Use `--stop-sequence "[/REVIEW]"` if you only want the review body.